In [ ]:
import os
import json
import pickle

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from datasets import load_dataset

In [ ]:
model_name = "gpt2"

layers_to_hook = [
    "h.0.mlp.c_fc",
    "h.0.mlp.c_proj",
    # add other layer names or indices as desired
]
save_path = "activation_cache.pkl"


device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
# load model & tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
ds = load_dataset("squad", split="validation[:20]")  # small demo

In [ ]:
class ActivationCache:
    def __init__(self, model, layers_to_hook=None):
        self.model = model
        self.layers_to_hook = layers_to_hook or []  # list of layer names or indices
        self.activations = {}  # dict: example_id → { layer_name → tensor }

    def _hook_fn(self, layer_name):
        def fn(module, input, output):
            # detach to move off graph, convert to CPU if needed
            self.activations[self.current_example_id][layer_name] = output.detach().cpu()
        return fn

    def register_hooks(self):
        for name, module in self.model.named_modules():
            if name in self.layers_to_hook:
                module.register_forward_hook(self._hook_fn(name))

    def clear(self):
        self.activations = {}

    def save(self, path):
        # save as pickle (alternatively use torch.save)
        with open(path, 'wb') as f:
            pickle.dump(self.activations, f)

In [ ]:
def collect_activations(
    texts,
    cache: ActivationCache,
    tokenizer,
    model,
    device=torch.device("cpu"),
    batch_size=1,
):
    model.eval()
    model.to(device)
    cache.register_hooks()
    for idx, text in enumerate(texts):
        cache.current_example_id = idx
        cache.activations[idx] = {}
        # tokenize
        inputs = tokenizer(text, return_tensors="pt", truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        # forward pass
        with torch.no_grad():
            _ = model(**inputs)
        # (activations stored via hooks)
    return cache.activations

In [ ]:
base = 0
for _ in range(0, len(ds), 100):
    texts = [f"Context: {ds[base+i]["context"]}\nQuestion: {ds[base+i]["question"]}\nAnswer:" for i in range(100)]
    # init cache
    cache = ActivationCache(model, layers_to_hook=layers_to_hook)

    # collect
    activations = collect_activations(texts, cache, tokenizer, model, device=device)

    # save
    cache.save(save_path)
    print(f"Saved activations for {len(texts)} examples to {save_path}")
    base += 100

In [5]:
import torch
import pickle
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from datasets import load_dataset
import os
import gc

model_name = "gpt2"
save_dir = "activation_cache"
os.makedirs(save_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Load model & tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
model.eval()

# Dataset
ds = load_dataset("squad", split="validation")

# --- Hook utility class ---
class ActivationCache:
    def __init__(self, model, layers_to_hook):
        self.model = model
        self.layers_to_hook = layers_to_hook
        self.activations = {}
        self._hooks = []

    def _hook_fn(self, layer_name):
        def fn(_, __, output):
            self.activations[self.current_example_id][layer_name] = output.detach().cpu()
        return fn

    def register_hooks(self):
        for name, module in self.model.named_modules():
            if name in self.layers_to_hook:
                self._hooks.append(module.register_forward_hook(self._hook_fn(name)))

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks = []

    def clear(self):
        self.activations.clear()
        gc.collect()
        torch.cuda.empty_cache()

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self.activations, f)

# --- collect activations ---
def collect_activations(texts, cache, tokenizer, model, device):
    cache.register_hooks()
    for idx, text in enumerate(texts):
        cache.current_example_id = idx
        cache.activations[idx] = {}
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            _ = model(**inputs)
    cache.remove_hooks()
    return cache.activations

# --- main loop ---
# Automatically find all MLP layers
layers_to_hook = [name for name, _ in model.named_modules() if "mlp.c_" in name]

chunk_size = 100
base = 0
for start in range(0, len(ds), chunk_size):
    end = min(start + chunk_size, len(ds))
    texts = [
        f"Context: {ds[i]['context']}\nQuestion: {ds[i]['question']}\nAnswer:"
        for i in range(start, end)
    ]

    cache = ActivationCache(model, layers_to_hook)
    activations = collect_activations(texts, cache, tokenizer, model, device)

    save_path = os.path.join(save_dir, f"activations_{start:05d}_{end:05d}.pkl")
    cache.save(save_path)
    print(f"Saved activations for examples {start}-{end} to {save_path}")

    cache.clear()

print("Done.")


cuda


/home/zkr/miniconda3/envs/internlm/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Saved activations for examples 0-100 to activation_cache/activations_00000_00100.pkl
Saved activations for examples 100-200 to activation_cache/activations_00100_00200.pkl
Saved activations for examples 200-300 to activation_cache/activations_00200_00300.pkl
Saved activations for examples 300-400 to activation_cache/activations_00300_00400.pkl
Saved activations for examples 400-500 to activation_cache/activations_00400_00500.pkl
Saved activations for examples 500-600 to activation_cache/activations_00500_00600.pkl
Saved activations for examples 600-700 to activation_cache/activations_00600_00700.pkl
Saved activations for examples 700-800 to activation_cache/activations_00700_00800.pkl
Saved activations for examples 800-900 to activation_cache/activations_00800_00900.pkl
Saved activations for examples 900-1000 to activation_cache/activations_00900_01000.pkl
Saved activations for examples 1000-1100 to activation_cache/activations_01000_01100.pkl
Saved activations for examples 1100-1200 t

OSError: [Errno 28] No space left on device

In [2]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from datasets import load_dataset

# model + tokenizer
model_name = "gpt2"           # or "distilgpt2" for smaller
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)
model.to("cuda")
# load small SQuAD validation subset
ds = load_dataset("squad", split="validation[:20]")  # small demo

def answer_with_gpt2(context, question, max_new_tokens=64):
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    # generate continuation (greedy). tune do_sample/top_k/top_p if you want variety.
    out = model.generate(input_ids, max_new_tokens=max_new_tokens, 
                         pad_token_id=tokenizer.eos_token_id, do_sample=False)
    generated = tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return generated.strip()

# Quick run
for i in range(5):
    item = ds[i]
    print("Q:", item["question"])
    print("-Pred--")
    print("Pred->:", answer_with_gpt2(item["context"], item["question"]))
    print("-Gold--")
    print("Gold:", item["answers"]["text"][0])
    print("---")


Q: Which NFL team represented the AFC at Super Bowl 50?
-Pred--
Pred->: The Denver Broncos.
Question: Which NFL team represented the NFC at Super Bowl 50?
Answer: The New York Giants.
Question: Which NFL team represented the NFC at Super Bowl 50?
Answer: The New York Jets.
Question: Which NFL team represented the NFC at Super Bowl 50?
Answer
-Gold--
Gold: Denver Broncos
---
Q: Which NFL team represented the NFC at Super Bowl 50?
-Pred--
Pred->: The Denver Broncos.
Question: Which NFL team represented the NFC at Super Bowl 50?
Answer: The Carolina Panthers.
Question: Which NFL team represented the NFC at Super Bowl 50?
Answer: The New York Giants.
Question: Which NFL team represented the NFC at Super Bowl 50?
Answer:
-Gold--
Gold: Carolina Panthers
---
Q: Where did Super Bowl 50 take place?
-Pred--
Pred->: The Super Bowl 50 celebration was held in the San Francisco Bay Area on February 7, 2016. The celebration was held in the San Francisco Bay Area on February 7, 2016. The celebration w